# Agente de gestión de operativos — CSA Salud Integral SpA (EP2)

**ISY0101 · Evaluación Parcial 2 · Leonel Gonzalez y Pablo Toro**

Continuación directa del prototipo RAG de la EP1. El asistente evoluciona a un **agente funcional**
que consulta, razona, planifica y escribe sobre un sistema de operativos **simulado**.

| Requisito EP2 | Implementación en este notebook |
|---|---|
| Herramientas de consulta, escritura y razonamiento (IE1) | 11 herramientas LangChain con *function calling* nativo |
| Framework de agentes (IE2) | **LangGraph `StateGraph`** para orquestación + LangChain tools + Mistral |
| Memoria de corto plazo (IE3) | Historial de mensajes LangChain + `trim_messages` (ventana de 3 turnos) |
| Memoria de largo plazo y recuperación semántica (IE3, IE4) | Índice FAISS persistente en Drive + herramienta `recordar` + estado de tarea pendiente en JSON |
| Planificación y decisiones adaptativas (IE5, IE7) | Grafo: preparar → router → consulta/gestión/fuera → finalizar; gestión con *Plan-and-Execute*, pausa y replanificación |

> Los procedimientos, operativos y consultas son ficticios. No se procesan datos clínicos ni personales reales.


## 1. Configuración

Antes de ejecutar: en **Secrets** de Colab crear `LLM_API_KEY` con la API key de Mistral y habilitar el acceso del notebook.
La celda monta Google Drive para que la memoria de largo plazo sobreviva al reinicio del entorno.

In [ ]:
!pip install -q -U langchain-core langchain-mistralai langchain-text-splitters langgraph faiss-cpu

In [ ]:
import os, json, re, time, copy
from datetime import datetime, date
from pathlib import Path
from typing import TypedDict, Any

import numpy as np
import pandas as pd
import faiss

from langchain_core.tools import tool
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, ToolMessage, trim_messages
)
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langgraph.graph import StateGraph, START, END

# --- Credenciales (Secrets de Colab o variable de entorno) ---
try:
    from google.colab import userdata
    LLM_API_KEY = userdata.get("LLM_API_KEY")
    EN_COLAB = True
except ImportError:
    LLM_API_KEY = os.getenv("LLM_API_KEY")
    EN_COLAB = False

if not LLM_API_KEY:
    raise ValueError("No se encontró LLM_API_KEY. Configúrala en los Secrets de Colab.")
LLM_API_KEY = LLM_API_KEY.strip()

# --- Persistencia: la memoria de largo plazo debe sobrevivir al reinicio de Colab ---
USAR_DRIVE = True
if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CSA_agente_EP2")
else:
    BASE_DIR = Path("./estado_agente")
BASE_DIR.mkdir(parents=True, exist_ok=True)
(BASE_DIR / "evidencias").mkdir(exist_ok=True)

# --- Parámetros del agente ---
MODELOS_CANDIDATOS = ["ministral-14b-2512", "ministral-8b-2512", "ministral-3b-2512"]
MODEL_EMBEDDING = "mistral-embed"
CHUNK_SIZE, CHUNK_OVERLAP = 450, 80     # validados en la EP1
UMBRAL_CORPUS = 0.78                    # validado en la EP1
UMBRAL_MEMORIA = 0.80                   # memoria de largo plazo: más estricto
TOP_K = 3
MAX_ITERACIONES = 5                     # límite por paso (evita bucles y costos)
MAX_PASOS_PLAN = 5
MAX_REPLANIFICACIONES = 2
VENTANA_CORTO_PLAZO = 6                 # mensajes (3 turnos)

# Un solo limitador para chat y embeddings (la capa gratuita de Mistral limita solicitudes por segundo)
limitador = InMemoryRateLimiter(requests_per_second=0.8, check_every_n_seconds=0.1, max_bucket_size=1)

def crear_llm(modelo, max_tokens=700):
    return ChatMistralAI(
        model=modelo, api_key=LLM_API_KEY, temperature=0,
        max_tokens=max_tokens, max_retries=3, timeout=60, rate_limiter=limitador,
    )

embedder = MistralAIEmbeddings(model=MODEL_EMBEDDING, api_key=LLM_API_KEY)

def _con_reintentos(fn, *args, intentos=4):
    for i in range(intentos):
        try:
            limitador.acquire()
            return fn(*args)
        except Exception as exc:
            if "429" in str(exc) and i < intentos - 1:
                time.sleep(2 * (i + 1))
                continue
            raise

def embed_textos(textos):
    return np.array(_con_reintentos(embedder.embed_documents, textos), dtype="float32")

def embed_consulta(texto):
    return np.array([_con_reintentos(embedder.embed_query, texto)], dtype="float32")

print("Entorno listo. Estado persistente en:", BASE_DIR)

## 2. Corpus documental (interno simulado + normativa pública)
Mismo corpus de la EP1: 6 procedimientos internos ficticios y 2 extractos de normativa chilena.

In [ ]:
def _doc(id_, titulo, tipo, origen, fuente, referencia, contenido):
    return {"id": id_, "titulo": titulo, "tipo": tipo, "origen": origen,
            "fuente": fuente, "referencia": referencia, "contenido": contenido.strip()}

FICTICIO = ("Procedimiento interno simulado", "interna_simulada",
            "CSA - Documento ficticio", "Documento ficticio con fines académicos")

documentos_internos = [
    _doc("INT-001", "Procedimiento para creación de un nuevo operativo", *FICTICIO, """
Para crear un nuevo operativo de salud, el colaborador responsable debe
registrar inicialmente los antecedentes generales del operativo,
incluyendo nombre de la empresa solicitante, fecha programada, ubicación,
tipo de servicio requerido y cantidad estimada de participantes.

Una vez registrados los antecedentes, el coordinador operativo debe
revisar que la información esté completa y asignar al personal requerido.

Posteriormente se debe confirmar la disponibilidad de recursos,
equipamiento y profesionales necesarios.

Una vez validada la información, el operativo cambia al estado
'Programado' y debe notificarse al responsable de coordinación."""),
    _doc("INT-002", "Procedimiento de carga de documentación de operativos", *FICTICIO, """
La documentación asociada a un operativo debe cargarse en el repositorio
correspondiente utilizando el identificador único del operativo.

Los documentos deben clasificarse según su tipo antes de ser almacenados.
Entre las categorías permitidas se encuentran documentación administrativa,
informes de ejecución, respaldos operativos y documentos de cierre.

Antes de realizar la carga, el colaborador debe verificar que el archivo
corresponda al operativo seleccionado y que su formato sea válido.

Una vez almacenado el documento, el sistema debe registrar la fecha de carga
y el usuario responsable para mantener trazabilidad."""),
    _doc("INT-003", "Procedimiento para modificación de un operativo", *FICTICIO, """
Cuando sea necesario modificar un operativo ya programado, el colaborador
debe ingresar al registro correspondiente y seleccionar la opción de edición.

Se podrán modificar antecedentes como fecha, ubicación, cantidad estimada de
participantes y recursos requeridos mientras el operativo no haya sido cerrado.

Los cambios considerados críticos deberán ser revisados por el coordinador
responsable antes de ser confirmados.

Cada modificación deberá quedar registrada en el historial del operativo,
indicando fecha, usuario responsable y descripción del cambio realizado."""),
    _doc("INT-004", "Procedimiento para cierre de un operativo", *FICTICIO, """
El cierre de un operativo se realiza una vez finalizadas las actividades
programadas y verificada la documentación correspondiente.

El responsable debe revisar que los registros administrativos se encuentren
completos y que los documentos requeridos hayan sido cargados correctamente.

Posteriormente deberá registrar una observación de cierre y cambiar el estado
del operativo a 'Finalizado'.

Una vez cerrado el operativo, las modificaciones posteriores deberán quedar
restringidas a usuarios autorizados."""),
    _doc("INT-005", "Protocolo de consulta de procedimientos operativos",
         "Protocolo interno simulado", *FICTICIO[1:], """
Los colaboradores pueden consultar procedimientos operativos utilizando
el asistente interno de información.

Las consultas deben formularse indicando claramente el proceso sobre el cual
se necesita orientación.

El asistente deberá responder utilizando únicamente la información disponible
en la base documental autorizada.

Cuando no exista información suficiente para responder una consulta, el
sistema deberá informarlo explícitamente y evitar generar instrucciones que
no estén respaldadas por una fuente documental."""),
    _doc("INT-006", "Gestión de incidencias durante un operativo", *FICTICIO, """
Cuando se produzca una incidencia durante un operativo, el colaborador
responsable debe registrar el evento indicando su descripción, hora de
ocurrencia y medidas adoptadas.

Las incidencias que afecten la continuidad del operativo deberán ser
informadas al coordinador responsable.

Cuando la incidencia sea resuelta, deberá actualizarse su estado y registrar
la acción que permitió solucionar el problema.

El registro de incidencias debe conservarse asociado al operativo para
facilitar su posterior revisión."""),
]

LEYCHILE = ("Normativa pública", "externa_publica", "Biblioteca del Congreso Nacional de Chile - LeyChile")

documentos_externos = [
    _doc("EXT-001", "Protección de datos personales - Ley 19.628", *LEYCHILE,
         "Ley N.º 19.628 sobre protección de la vida privada", """
La legislación chilena sobre protección de la vida privada regula el
tratamiento de datos personales almacenados en registros o bancos de datos.

Se consideran datos personales aquellas informaciones relacionadas con
personas naturales identificadas o identificables.

Los datos sensibles comprenden información relacionada con aspectos de la
vida privada de las personas, incluyendo antecedentes vinculados con su
estado de salud físico o psíquico.

El tratamiento de datos personales debe realizarse conforme a la legislación
aplicable y respetando los derechos de los titulares de los datos."""),
    _doc("EXT-002", "Condiciones sanitarias y ambientales en lugares de trabajo", *LEYCHILE,
         "Decreto Supremo N.º 594 del Ministerio de Salud", """
El Decreto Supremo N.º 594 establece condiciones sanitarias y ambientales
básicas que deben cumplir los lugares de trabajo.

La normativa establece que las empresas deben mantener condiciones
sanitarias y ambientales necesarias para proteger la vida y salud de los
trabajadores.

Los lugares de trabajo deben mantenerse en condiciones adecuadas de orden
y limpieza y deben considerar condiciones de seguridad y saneamiento
compatibles con las actividades desarrolladas.

La normativa también contempla materias relacionadas con ventilación,
servicios higiénicos y exposición de trabajadores a determinados agentes
ambientales."""),
]

documentos_totales = documentos_internos + documentos_externos
print("Documentos internos:", len(documentos_internos), "| externos:", len(documentos_externos))

## 3. Recuperación semántica: índice vectorial FAISS
Chunking 450/80 y embedding enriquecido con el título (decisiones validadas en la EP1). La misma clase se usa para el corpus y para la memoria de largo plazo.

In [ ]:
class IndiceVectorial:
    """Índice FAISS (similitud coseno) con metadatos. Se usa para el corpus y para la memoria de largo plazo."""

    def __init__(self, fn_docs=None, fn_query=None):
        self.fn_docs = fn_docs or embed_textos
        self.fn_query = fn_query or embed_consulta
        self.index = None
        self.registros = []

    def agregar(self, textos_embedding, registros):
        vectores = self.fn_docs(textos_embedding)
        faiss.normalize_L2(vectores)
        if self.index is None:
            self.index = faiss.IndexFlatIP(vectores.shape[1])
        self.index.add(vectores)
        self.registros.extend(registros)

    def buscar(self, consulta, top_k=TOP_K, umbral=UMBRAL_CORPUS, filtro=None):
        if self.index is None or self.index.ntotal == 0:
            return []
        q = self.fn_query(consulta)
        faiss.normalize_L2(q)
        k = min(max(top_k * 4, top_k), self.index.ntotal)
        scores, ids = self.index.search(q, k)
        salida = []
        for score, i in zip(scores[0], ids[0]):
            if i < 0:
                continue
            registro = self.registros[i]
            if filtro and not filtro(registro):
                continue
            if float(score) < umbral:
                continue
            salida.append({**registro, "score": round(float(score), 4)})
            if len(salida) >= top_k:
                break
        return salida

    def guardar(self, carpeta):
        carpeta = Path(carpeta)
        carpeta.mkdir(parents=True, exist_ok=True)
        if self.index is not None:
            faiss.write_index(self.index, str(carpeta / "indice.faiss"))
        (carpeta / "registros.json").write_text(
            json.dumps(self.registros, ensure_ascii=False, indent=1), encoding="utf-8")

    @classmethod
    def cargar(cls, carpeta, **kwargs):
        carpeta = Path(carpeta)
        obj = cls(**kwargs)
        if (carpeta / "indice.faiss").exists():
            obj.index = faiss.read_index(str(carpeta / "indice.faiss"))
            obj.registros = json.loads((carpeta / "registros.json").read_text(encoding="utf-8"))
        return obj


# --- Corpus: chunking enriquecido con título (decisión validada en la EP1) ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""])

def crear_chunks(documentos):
    chunks = []
    for d in documentos:
        for i, frag in enumerate(splitter.split_text(d["contenido"])):
            chunks.append({
                "chunk_id": f'{d["id"]}-CH{i + 1:02d}', "documento_id": d["id"],
                "titulo": d["titulo"], "origen": d["origen"], "referencia": d["referencia"],
                "chunk_index": i, "contenido": frag,
                "texto_embedding": f'Código: {d["id"]}\nTítulo: {d["titulo"]}\nTipo de fuente: {d["origen"]}\n\n{frag}',
            })
    return chunks

def construir_indice_corpus():
    chunks = crear_chunks(documentos_totales)
    indice = IndiceVectorial()
    indice.agregar([c["texto_embedding"] for c in chunks],
                   [{k: v for k, v in c.items() if k != "texto_embedding"} for c in chunks])
    return indice

indice_corpus = construir_indice_corpus()
print("Chunks indexados:", indice_corpus.index.ntotal)

## 4. Memoria de largo plazo y estado de tareas
- **Memoria de largo plazo:** las interacciones verificables se consolidan en un índice FAISS persistente.
  La memoria **no se inyecta automáticamente en acciones de escritura**; se consulta explícitamente
  mediante la herramienta `recordar`, evitando que recuerdos semánticamente parecidos contaminen
  los datos de otro operativo.
- **Estado de tarea:** si un flujo de varias etapas se pausa por datos faltantes, el plan queda guardado
  para retomarlo en el turno o sesión siguiente.


In [ ]:
class MemoriaLargoPlazo:
    """Memoria persistente entre sesiones: registros de interacciones, notas y tareas pendientes.
    Se recupera por similitud semántica (embeddings + FAISS) y se guarda en disco/Drive."""

    def __init__(self, carpeta):
        self.carpeta = Path(carpeta)
        self.indice = IndiceVectorial.cargar(self.carpeta)

    def guardar(self, texto, tipo="interaccion", **meta):
        registro = {"tipo": tipo, "texto": texto, "orden": self.total() + 1,
                    "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), **meta}
        self.indice.agregar([texto], [registro])
        self.indice.guardar(self.carpeta)
        return registro

    def recuperar(self, consulta, top_k=2, umbral=UMBRAL_MEMORIA, tipo=None):
        """Recupera memoria semántica priorizando estados finales verificados.

        Se sobre-muestrean candidatos para evitar que un recuerdo intermedio muy
        parecido desplace al registro final de la misma tarea.
        """
        filtro = (lambda r: r["tipo"] == tipo) if tipo else None
        candidatos = self.indice.buscar(
            consulta,
            top_k=max(top_k * 3, 8),
            umbral=umbral,
            filtro=filtro
        )

        prioridad_tipo = {
            "tarea_resuelta": 5,
            "nota": 4,
            "interaccion": 3,
            "pendiente_resuelto": 2,
            "pendiente": 1,
        }

        candidatos = sorted(
            candidatos,
            key=lambda r: (
                prioridad_tipo.get(r.get("tipo"), 0),
                r.get("orden", 0)
            ),
            reverse=True
        )

        # Si existe una resolución explícita de una tarea, se eliminan de la
        # respuesta sus estados pendientes/intermedios anteriores.
        tareas_resueltas = {
            r.get("tarea_id")
            for r in candidatos
            if r.get("tipo") == "tarea_resuelta" and r.get("tarea_id")
        }

        filtrados = []
        for r in candidatos:
            if (
                r.get("tarea_id") in tareas_resueltas
                and r.get("tipo") in {"pendiente", "pendiente_resuelto"}
            ):
                continue
            filtrados.append(r)

        return filtrados[:top_k]

    def resolver_pendientes(self, tarea_id: str):
        """Marca como resuelto únicamente el pendiente asociado a `tarea_id`.

        Evita que una acción exitosa sobre otro operativo cierre por accidente
        una tarea distinta que todavía estaba esperando datos del usuario.
        """
        if not tarea_id:
            return 0

        cambios = 0
        for r in self.indice.registros:
            if r.get("tipo") == "pendiente" and r.get("tarea_id") == tarea_id:
                r["tipo"] = "pendiente_resuelto"
                r["texto"] += " | ESTADO: RESUELTO en un turno posterior"
                cambios += 1

        if cambios:
            self.indice.guardar(self.carpeta)
        return cambios

    def total(self):
        return 0 if self.indice.index is None else self.indice.index.ntotal


class EstadoTarea:
    """Tarea de varias etapas que quedó pausada (por ejemplo, por datos faltantes). Persiste en JSON."""

    def __init__(self, ruta):
        self.ruta = Path(ruta)

    def leer(self):
        return json.loads(self.ruta.read_text(encoding="utf-8")) if self.ruta.exists() else None

    def guardar(self, tarea):
        self.ruta.write_text(json.dumps(tarea, ensure_ascii=False, indent=1), encoding="utf-8")

    def limpiar(self):
        if self.ruta.exists():
            self.ruta.unlink()


memoria_lp = MemoriaLargoPlazo(BASE_DIR / "memoria_largo_plazo")
estado_tarea = EstadoTarea(BASE_DIR / "tarea_pendiente.json")
print("Registros en memoria de largo plazo:", memoria_lp.total())

## 5. Sistema simulado de operativos
Base JSON con tres operativos de ejemplo en distintos estados: `Programado`, `Finalizado` y `Registrado` con datos incompletos. Las reglas de negocio provienen de los procedimientos INT-001, INT-003, INT-004 e INT-006.

In [ ]:
RUTA_OPERATIVOS = BASE_DIR / "operativos.json"

OPERATIVOS_INICIALES = {
    "OP-001": {"id": "OP-001", "empresa": "Empresa Andes Ltda.", "fecha": "2026-12-03",
               "ubicacion": "Providencia, Santiago", "tipo_servicio": "Exámenes preventivos",
               "participantes": 40, "coordinador": "Coordinador A (ficticio)",
               "recursos_confirmados": True, "estado": "Programado",
               "documentos": ["Documentación administrativa"], "historial": [], "incidencias": []},
    "OP-002": {"id": "OP-002", "empresa": "Constructora Sur SpA", "fecha": "2026-09-15",
               "ubicacion": "San Bernardo", "tipo_servicio": "Vacunación",
               "participantes": 25, "coordinador": "Coordinador B (ficticio)",
               "recursos_confirmados": True, "estado": "Finalizado",
               "documentos": ["Documentación administrativa", "Documentos de cierre"],
               "historial": [], "incidencias": [], "observacion_cierre": "Actividades completadas."},
    "OP-003": {"id": "OP-003", "empresa": "Minera Norte S.A.", "fecha": "2026-12-10",
               "ubicacion": "Antofagasta", "tipo_servicio": None, "participantes": 60,
               "coordinador": None, "recursos_confirmados": False, "estado": "Registrado",
               "documentos": [], "historial": [], "incidencias": []},
}

CAMPOS_OBLIGATORIOS = ["empresa", "fecha", "ubicacion", "tipo_servicio", "participantes"]  # INT-001
CAMPOS_EDITABLES = ["fecha", "ubicacion", "participantes", "tipo_servicio", "recursos_confirmados"]  # INT-003
CAMPOS_CRITICOS = ["fecha", "ubicacion"]


def cargar_operativos():
    if not RUTA_OPERATIVOS.exists():
        guardar_operativos(copy.deepcopy(OPERATIVOS_INICIALES))
    return json.loads(RUTA_OPERATIVOS.read_text(encoding="utf-8"))

def guardar_operativos(datos):
    RUTA_OPERATIVOS.write_text(json.dumps(datos, ensure_ascii=False, indent=1), encoding="utf-8")

def reiniciar_datos_demo():
    """Deja el sistema simulado en su estado inicial (no borra la memoria de largo plazo)."""
    guardar_operativos(copy.deepcopy(OPERATIVOS_INICIALES))

def _registrar_historial(op, accion):
    op["historial"].append({"fecha": datetime.now().strftime("%Y-%m-%d %H:%M"),
                            "usuario": "agente (simulado)", "accion": accion})

def _faltantes(op):
    return [c for c in CAMPOS_OBLIGATORIOS if op.get(c) in (None, "", 0)]

def _validar_fecha(valor):
    try:
        f = datetime.strptime(str(valor), "%Y-%m-%d").date()
    except ValueError:
        return "formato de fecha inválido (usar AAAA-MM-DD)"
    if f <= date.today():
        return "la fecha debe ser posterior a hoy"
    return None

PATRON_ID_OPERATIVO = re.compile(r"^OP-\d{3,}$", re.IGNORECASE)

def _normalizar_id_operativo(valor):
    """Normaliza un ID y devuelve None si no corresponde al formato OP-XXX."""
    valor = str(valor or "").strip().upper()
    return valor if PATRON_ID_OPERATIVO.fullmatch(valor) else None

def _respuesta_id_operativo_invalido(valor):
    """Respuesta estándar para evitar confundir IDs de documentos INT-* con operativos OP-*."""
    return _json(
        estado="rechazado",
        reintentable=True,
        detalle=(
            f"Identificador de operativo inválido: {valor!r}. "
            "Los operativos usan formato OP-XXX. Los identificadores INT-XXX "
            "corresponden a documentos/procedimientos y no pueden usarse como operativos."
        )
    )

# Datos que no deben quedar en registros operativos (Ley 19.628: datos sensibles de salud / identificación)
PATRON_SENSIBLE = re.compile(
    r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b|diagn[oó]stic|enfermedad|medicamento|receta|"
    r"patolog|ficha cl[ií]nica|tratamiento m[eé]dico|diabet|hipertens|epilep|embaraz|vih\b|c[aá]ncer|"
    r"alergi|psiqui|depresi|estado de salud|condici[oó]n (m[eé]dica|preexistente|de salud|cr[oó]nica)",
    re.IGNORECASE)

def _json(**kwargs):
    return json.dumps(kwargs, ensure_ascii=False)

cargar_operativos()
print("Operativos simulados:", list(cargar_operativos()))

## 6. Herramientas del agente

| Tipo | Herramientas |
|---|---|
| Consulta | `buscar_procedimientos`, `buscar_normativa`, `consultar_operativo`, `listar_operativos` |
| Razonamiento | `validar_requisitos` (reglas de INT-001: pendientes y siguiente paso) |
| Escritura | `crear_operativo`, `programar_operativo`, `actualizar_operativo`, `registrar_incidencia` |
| Memoria | `recordar`, `guardar_nota` |

Todas devuelven JSON con un campo `estado` (`ok`, `faltan_datos`, `rechazado`, `error`, `sin_evidencia`). El orquestador usa ese campo para decidir si continúa, pausa o replanifica.

In [ ]:
def _formatear_resultados(resultados):
    return [{"fuente": r["titulo"], "chunk": r["chunk_id"], "referencia": r["referencia"],
             "score": r["score"], "contenido": r["contenido"]} for r in resultados]

# ---------------------------- HERRAMIENTAS DE CONSULTA ----------------------------
@tool
def buscar_procedimientos(consulta: str) -> str:
    """Busca en los procedimientos internos de CSA (creación, documentación, modificación,
    cierre e incidencias de operativos). Úsala antes de explicar o ejecutar un procedimiento."""
    res = indice_corpus.buscar(consulta, filtro=lambda r: r["origen"] == "interna_simulada")
    if not res:
        return _json(estado="sin_evidencia", detalle="La búsqueda no encontró fragmentos sobre el umbral. No significa que el procedimiento no exista: reformula la consulta con palabras del tema (crear, cerrar, modificar, incidencia).")
    return _json(estado="ok", resultados=_formatear_resultados(res))

@tool
def buscar_normativa(consulta: str) -> str:
    """Busca en la normativa pública chilena incorporada al corpus (Ley 19.628 de datos
    personales y DS 594 de condiciones sanitarias en lugares de trabajo)."""
    res = indice_corpus.buscar(consulta, filtro=lambda r: r["origen"] == "externa_publica")
    if not res:
        return _json(estado="sin_evidencia", detalle="La búsqueda no encontró normativa sobre el umbral. No significa que no exista: reformula la consulta con palabras del tema (datos personales, condiciones sanitarias).")
    return _json(estado="ok", resultados=_formatear_resultados(res))

@tool
def consultar_operativo(id_operativo: str) -> str:
    """Devuelve el registro completo de un operativo (estado, datos, historial e incidencias).
    El identificador tiene el formato OP-XXX."""
    id_normalizado = _normalizar_id_operativo(id_operativo)
    if not id_normalizado:
        return _respuesta_id_operativo_invalido(id_operativo)
    op = cargar_operativos().get(id_normalizado)
    if not op:
        return _json(estado="error", detalle=f"No existe el operativo {id_normalizado}.")
    return _json(estado="ok", operativo=op)

@tool
def listar_operativos(estado_filtro: str = "") -> str:
    """Lista los operativos con su estado. Opcionalmente filtra por estado:
    Registrado, Programado o Finalizado."""
    ops = cargar_operativos().values()
    if estado_filtro:
        ops = [o for o in ops if o["estado"].lower() == estado_filtro.strip().lower()]
    return _json(estado="ok", operativos=[
        {"id": o["id"], "empresa": o["empresa"], "fecha": o["fecha"], "estado": o["estado"]} for o in ops])

# --------------------------- HERRAMIENTA DE RAZONAMIENTO ---------------------------
@tool
def validar_requisitos(id_operativo: str) -> str:
    """Evalúa si un operativo cumple los requisitos de INT-001 para pasar a 'Programado':
    antecedentes completos, fecha válida, coordinador asignado y recursos confirmados.
    Devuelve los pendientes y el siguiente paso recomendado."""
    id_normalizado = _normalizar_id_operativo(id_operativo)
    if not id_normalizado:
        return _respuesta_id_operativo_invalido(id_operativo)
    op = cargar_operativos().get(id_normalizado)
    if not op:
        return _json(estado="error", detalle=f"No existe el operativo {id_normalizado}.")
    if op["estado"] == "Finalizado":
        return _json(estado="rechazado", reintentable=False, detalle="El operativo está cerrado (INT-004).")
    pendientes = [f"falta {c}" for c in _faltantes(op)]
    error_fecha = _validar_fecha(op["fecha"]) if op.get("fecha") else None
    if error_fecha:
        pendientes.append(error_fecha)
    if not op.get("coordinador"):
        pendientes.append("falta asignar coordinador")
    if not op.get("recursos_confirmados"):
        pendientes.append("falta confirmar recursos, equipamiento y profesionales")
    if pendientes:
        siguiente = ("completar antecedentes" if _faltantes(op) or error_fecha
                     else "asignar coordinador y confirmar recursos")
        return _json(estado="faltan_datos", id=op["id"], pendientes=pendientes,
                     siguiente_paso=siguiente, fuente="INT-001")
    return _json(estado="ok", id=op["id"], detalle="Cumple requisitos para 'Programado'.", fuente="INT-001")

# ----------------------------- HERRAMIENTAS DE ESCRITURA -----------------------------
@tool
def crear_operativo(empresa: str = "", fecha: str = "", ubicacion: str = "",
                    tipo_servicio: str = "", participantes: int = 0) -> str:
    """Registra un nuevo operativo en estado 'Registrado' (INT-001). Requiere empresa,
    fecha (AAAA-MM-DD), ubicacion, tipo_servicio y participantes. No inventes valores:
    si el usuario no los entregó, deja el campo vacío para que la herramienta los reporte."""
    datos = {"empresa": empresa.strip(), "fecha": fecha.strip(), "ubicacion": ubicacion.strip(),
             "tipo_servicio": tipo_servicio.strip(), "participantes": int(participantes or 0)}
    faltan = [c for c in CAMPOS_OBLIGATORIOS if not datos[c]]
    if faltan:
        return _json(estado="faltan_datos", pendientes=faltan,
                     detalle="No se registró el operativo: faltan antecedentes obligatorios (INT-001).")
    error_fecha = _validar_fecha(datos["fecha"])
    if error_fecha:
        return _json(estado="rechazado", reintentable=False, detalle=f"{error_fecha}; pedir una fecha válida al usuario")
    ops = cargar_operativos()
    nuevo_id = f"OP-{len(ops) + 1:03d}"
    ops[nuevo_id] = {"id": nuevo_id, **datos, "coordinador": None, "recursos_confirmados": False,
                     "estado": "Registrado", "documentos": [], "historial": [], "incidencias": []}
    _registrar_historial(ops[nuevo_id], "Operativo registrado")
    guardar_operativos(ops)
    return _json(estado="ok", id=nuevo_id, nuevo_estado="Registrado",
                 siguiente_paso="asignar coordinador y confirmar recursos (INT-001)")

@tool
def programar_operativo(id_operativo: str, coordinador: str = "", recursos_confirmados: bool = False) -> str:
    """Asigna coordinador, registra la confirmación de recursos y cambia el operativo a
    'Programado' si cumple todos los requisitos (INT-001). Registra la notificación al
    responsable de coordinación. Solo usar si el usuario indicó coordinador y confirmó recursos."""
    id_normalizado = _normalizar_id_operativo(id_operativo)
    if not id_normalizado:
        return _respuesta_id_operativo_invalido(id_operativo)
    ops = cargar_operativos()
    op = ops.get(id_normalizado)
    if not op:
        return _json(estado="error", detalle=f"No existe el operativo {id_normalizado}.")
    if op["estado"] == "Finalizado":
        return _json(estado="rechazado", reintentable=False, detalle="El operativo está cerrado (INT-004).")
    if coordinador.strip():
        op["coordinador"] = coordinador.strip()
    if recursos_confirmados:
        op["recursos_confirmados"] = True
    pendientes = [f"falta {c}" for c in _faltantes(op)]
    if not op.get("coordinador"):
        pendientes.append("falta asignar coordinador")
    if not op.get("recursos_confirmados"):
        pendientes.append("falta confirmar recursos")
    guardar_operativos(ops)
    if pendientes:
        return _json(estado="faltan_datos", id=op["id"], pendientes=pendientes)
    op["estado"] = "Programado"
    _registrar_historial(op, "Estado cambiado a Programado; notificación al responsable de coordinación (simulada)")
    guardar_operativos(ops)
    return _json(estado="ok", id=op["id"], nuevo_estado="Programado",
                 detalle="Notificación al responsable de coordinación registrada (simulada).")

@tool
def actualizar_operativo(id_operativo: str, campo: str, valor: str, motivo: str = "") -> str:
    """Modifica un antecedente de un operativo no cerrado (INT-003). Campos permitidos:
    fecha, ubicacion, participantes, tipo_servicio, recursos_confirmados.
    Los cambios de fecha o ubicación quedan marcados para revisión del coordinador."""
    id_normalizado = _normalizar_id_operativo(id_operativo)
    if not id_normalizado:
        return _respuesta_id_operativo_invalido(id_operativo)
    ops = cargar_operativos()
    op = ops.get(id_normalizado)
    campo = campo.strip().lower()
    if not op:
        return _json(estado="error", detalle=f"No existe el operativo {id_normalizado}.")
    if op["estado"] == "Finalizado":
        return _json(
            estado="rechazado",
            reintentable=False,
            detalle=(
                "INT-003 permite modificar antecedentes mientras el operativo no haya sido cerrado. "
                f"{id_normalizado} está Finalizado, por lo que la modificación se rechaza. "
                "INT-004 además restringe modificaciones posteriores al cierre a usuarios autorizados."
            ),
            fuente="INT-003 / INT-004"
        )
    if campo not in CAMPOS_EDITABLES:
        return _json(estado="rechazado", reintentable=False, detalle=f"Campo no editable. Permitidos: {CAMPOS_EDITABLES}")
    if campo == "fecha":
        error_fecha = _validar_fecha(valor)
        if error_fecha:
            return _json(estado="rechazado", reintentable=False, detalle=f"{error_fecha}; pedir una fecha válida al usuario")
    if campo == "participantes":
        valor = int(valor)
    if campo == "recursos_confirmados":
        valor = str(valor).lower() in ("true", "si", "sí", "1")
    anterior = op.get(campo)
    op[campo] = valor
    critico = campo in CAMPOS_CRITICOS
    _registrar_historial(op, f"{campo}: {anterior} -> {valor}. Motivo: {motivo or 'no indicado'}"
                         + (" [requiere revisión del coordinador]" if critico else ""))
    guardar_operativos(ops)
    return _json(estado="ok", id=op["id"], campo=campo, anterior=anterior, nuevo=valor,
                 requiere_revision_coordinador=critico, fuente="INT-003")

@tool
def registrar_incidencia(id_operativo: str, descripcion: str, hora: str, medidas: str,
                         afecta_continuidad: bool = False) -> str:
    """Registra una incidencia ocurrida DURANTE la ejecución de un operativo (INT-006): descripción,
    hora (HH:MM) y medidas adoptadas. No sirve para registrar solicitudes de cambio.
    No debe contener datos personales ni de salud de personas."""
    id_normalizado = _normalizar_id_operativo(id_operativo)
    if not id_normalizado:
        return _respuesta_id_operativo_invalido(id_operativo)
    ops = cargar_operativos()
    op = ops.get(id_normalizado)
    if not op:
        return _json(estado="error", detalle=f"No existe el operativo {id_normalizado}.")
    if op["estado"] == "Finalizado":
        return _json(estado="rechazado", reintentable=False, detalle="Operativo cerrado (INT-004).")
    texto = f"{descripcion} {medidas}"
    if PATRON_SENSIBLE.search(texto):
        return _json(estado="rechazado", reintentable=True,
                     detalle="La incidencia contiene posibles datos sensibles de salud o identificación "
                             "(Ley 19.628). Reformular sin identificar personas ni su estado de salud.")
    faltan = [n for n, v in (("descripcion", descripcion), ("hora", hora), ("medidas", medidas)) if not v.strip()]
    if faltan:
        return _json(estado="faltan_datos", pendientes=faltan)
    registro = {"descripcion": descripcion, "hora": hora, "medidas": medidas,
                "afecta_continuidad": afecta_continuidad, "estado": "Abierta"}
    op["incidencias"].append(registro)
    _registrar_historial(op, "Incidencia registrada" + (" e informada al coordinador" if afecta_continuidad else ""))
    guardar_operativos(ops)
    return _json(estado="ok", id=op["id"], registro=registro, total_incidencias=len(op["incidencias"]),
                 informar_coordinador=afecta_continuidad, fuente="INT-006")

# ------------------------------ HERRAMIENTAS DE MEMORIA ------------------------------
@tool
def recordar(consulta: str) -> str:
    """Busca memoria persistente. Los estados finales verificados prevalecen
    sobre pendientes o estados intermedios de la misma tarea."""
    res = memoria_lp.recuperar(
        consulta,
        top_k=4,
        umbral=UMBRAL_MEMORIA - 0.08
    )
    if not res:
        return _json(
            estado="sin_evidencia",
            detalle="No hay recuerdos relevantes."
        )

    return _json(
        estado="ok",
        nota=(
            "Los estados finales verificados de una tarea prevalecen sobre "
            "sus estados pendientes o intermedios anteriores."
        ),
        recuerdos=[
            {
                "orden": r.get("orden"),
                "fecha": r["fecha"],
                "tipo": r["tipo"],
                "tarea_id": r.get("tarea_id"),
                "entidad": r.get("entidad"),
                "texto": r["texto"]
            }
            for r in res
        ]
    )

@tool
def guardar_nota(nota: str) -> str:
    """Guarda en la memoria de largo plazo un dato relevante para sesiones futuras
    (acuerdos, pendientes, preferencias). No guardar datos personales ni de salud."""
    if PATRON_SENSIBLE.search(nota):
        return _json(estado="rechazado", reintentable=True, detalle="La nota contiene posibles datos sensibles (Ley 19.628).")
    memoria_lp.guardar(nota, tipo="nota")
    return _json(estado="ok", detalle="Nota guardada en memoria de largo plazo.")


HERRAMIENTAS_CONSULTA = [buscar_procedimientos, buscar_normativa, consultar_operativo, listar_operativos, recordar]
HERRAMIENTAS_GESTION = [validar_requisitos, crear_operativo, programar_operativo,
                        actualizar_operativo, registrar_incidencia, guardar_nota]
HERRAMIENTAS = {t.name: t for t in HERRAMIENTAS_CONSULTA + HERRAMIENTAS_GESTION}

print(f"{len(HERRAMIENTAS)} herramientas:", ", ".join(HERRAMIENTAS))

## 7. Selección del modelo
Se prueba cada modelo candidato y se usa el primero que esté disponible en la cuenta y ejecute *function calling*. En la EP1, `mistral-small` devolvió cuota 0, por eso no se incluye.

In [ ]:
def seleccionar_modelo(candidatos=MODELOS_CANDIDATOS):
    """Elige el primer modelo disponible en la cuenta que además ejecute Function Calling."""
    for modelo in candidatos:
        try:
            respuesta = crear_llm(modelo, 80).bind_tools([listar_operativos]).invoke(
                "Lista los operativos en estado Programado usando la herramienta.")
            if respuesta.tool_calls:
                print(f"✔ {modelo}: disponible y con function calling.")
                return modelo
            print(f"✘ {modelo}: responde, pero no llamó a la herramienta.")
        except Exception as exc:
            print(f"✘ {modelo}: {str(exc)[:120]}")
    raise RuntimeError("Ningún modelo candidato quedó disponible. Revisa la API key o la cuota.")

MODEL_LLM = seleccionar_modelo()
llm = crear_llm(MODEL_LLM)

## 8. Agente orquestador con LangGraph

La lógica existente se conserva, pero la orquestación ahora está representada explícitamente mediante
**LangGraph `StateGraph`**:

1. **Preparar:** carga tarea pendiente y contexto seguro.
2. **Router:** clasifica en `CONSULTA`, `GESTION` o `FUERA`.
3. **Consulta:** ejecuta ciclo ReAct con herramientas de lectura y memoria.
4. **Gestión:** planifica, ejecuta tools de forma secuencial, pausa ante datos faltantes y replantea
   cuando una condición reintentable cambia.
5. **Finalizar:** consolida únicamente hechos verificables y actualiza la memoria de corto/largo plazo.

La memoria semántica de largo plazo se mantiene fuera del contexto de escritura. Para modificar un
operativo, el agente debe consultar su estado real mediante herramientas como `consultar_operativo`.


In [ ]:
PROMPT_SISTEMA = """Eres el Agente de Gestión de Operativos de CSA Salud Integral SpA (prototipo académico).
Ayudas a colaboradores a consultar procedimientos y a gestionar operativos en un sistema SIMULADO.

REGLAS OBLIGATORIAS:
1. Los resultados de las herramientas son la única fuente factual. No inventes procedimientos,
   fechas, responsables, identificadores ni datos.
2. Antes de explicar un procedimiento, consúltalo con buscar_procedimientos o buscar_normativa.
3. Nunca completes con suposiciones los datos de una herramienta de escritura: si el usuario no
   entregó un dato, déjalo vacío y la herramienta indicará lo que falta.
4. Si una herramienta responde "rechazado" o "faltan_datos", no insistas: explica el motivo.
5. No solicites, registres ni generes datos personales o clínicos reales, y no repitas en tu respuesta
   datos de salud que haya mencionado el usuario.
6. Responde en español, directo y en máximo 150 palabras, sin encabezados como "Respuesta para el
   usuario" ni formato de correo. Indica la fuente (documento o chunk) usada.
7. El historial y la memoria de largo plazo sirven para entender el contexto; no son fuente normativa
   y NUNCA se usan para completar datos obligatorios de una escritura (solo cuentan los datos que
   entrega el usuario en este turno o los de la tarea pendiente).
8. No atribuyas a una norma o procedimiento algo que su fragmento no dice. No agregues requisitos que las herramientas no indicaron. Describe los datos registrados tal como
   aparecen en los resultados; no afirmes omisiones ni cumplimientos que no estén en ellos.
9. La memoria de largo plazo solo puede consultarse explícitamente con la herramienta recordar. Nunca uses un recuerdo para completar argumentos de herramientas de escritura.
10. Distingue identificadores: INT-XXX identifica documentos/procedimientos; OP-XXX identifica operativos.
    Nunca pases un INT-XXX como argumento id_operativo.
11. Al explicar INT-003, conserva su condición exacta: los antecedentes pueden modificarse
    mientras el operativo no haya sido cerrado. No inventes estados como "activo" ni afirmes
    que solo un estado específico permite modificar.
"""

PROMPT_ROUTER = """Clasifica el mensaje del usuario en EXACTAMENTE una etiqueta:
CONSULTA = pregunta sobre procedimientos, normativa, estado de operativos o sesiones anteriores.
GESTION = pide realizar acciones (crear, programar, modificar, registrar incidencias) o entrega
          datos para continuar una tarea pendiente.
FUERA = no corresponde a operativos de CSA o pide consultar datos clínicos o personales de personas.

Importante: las preguntas sobre cómo se hace algo ("¿qué pasos debo seguir para...?", "¿cómo se...?")
son CONSULTA aunque mencionen una acción. Si el usuario pide registrar una incidencia o gestionar un operativo, es GESTION aunque
mencione temas de salud; las herramientas se encargan de filtrar los datos sensibles.

Tarea pendiente: {pendiente}
Historial reciente:
{historial}

Mensaje: {mensaje}
Responde solo la etiqueta."""

PROMPT_PLANIFICADOR = """Eres el planificador del agente. Descompón la solicitud en pasos ejecutables.

Herramientas disponibles:
{catalogo}

Reglas de planificación:
- Máximo {max_pasos} pasos, en orden de ejecución. Cada "accion" en máximo 15 palabras.
- validar_requisitos solo informa pendientes; si el usuario ya entregó esos datos, el paso
  siguiente debe usarlos (por ejemplo, programar_operativo con el coordinador indicado).
- Primero consulta (procedimiento o estado del operativo), luego valida, al final escribe.
- Nunca planifiques una escritura sin haber validado o consultado antes.
- Prioridad "alta" = imprescindible para el objetivo; "media" = recomendable; "baja" = opcional.
- Usa solo nombres de herramientas de la lista.
- Si hay observaciones de un rechazo, NO repitas la acción rechazada: planifica una alternativa
  (consultar el procedimiento o la normativa, corregir los datos o informar la restricción).
- Usa cada herramienta de escritura una sola vez por plan.
- Los antecedentes de memoria NO son datos de esta solicitud: no los uses para completar campos.
- En una tarea de GESTION no planifiques la herramienta recordar; para estado real usa consultar_operativo.
- Usa guardar_nota solo si el usuario pide explícitamente recordar, anotar o guardar una nota.
- Usa buscar_normativa solo si la solicitud o una observación menciona normativa, privacidad, datos sensibles, seguridad o regulación.
- Si hay una tarea pendiente y el usuario entrega los datos faltantes, retómala con todos los datos.
- INT-XXX es un identificador documental; las herramientas con argumento id_operativo aceptan únicamente OP-XXX.

Tarea pendiente previa: {pendiente}
Antecedentes de memoria: {memoria}
Observaciones de una ejecución anterior (si las hay): {observaciones}

Solicitud del usuario: {mensaje}

Devuelve SOLO JSON válido, sin Markdown:
{{"objetivo": "...", "pasos": [{{"n": 1, "accion": "...", "herramientas": ["..."], "prioridad": "alta"}}]}}"""


def _texto(contenido):
    if isinstance(contenido, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in contenido)
    return str(contenido)

def _extraer_json(texto):
    limpio = re.sub(r"^```(?:json)?\s*|\s*```$", "", _texto(texto).strip(), flags=re.IGNORECASE)
    try:
        return json.loads(limpio)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", limpio, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise

def _estado_resultado(salida):
    try:
        return json.loads(salida).get("estado", "ok")
    except (json.JSONDecodeError, AttributeError):
        return "ok"


# Regla determinista previa al router: preguntas informativas sin verbo imperativo -> CONSULTA
PATRON_PREGUNTA = re.compile(r"^\s*¿\s*(qu[eé]|c[oó]mo|cu[aá]l|cu[aá]les|d[oó]nde|cu[aá]ndo|debo|se puede|existe)\b", re.IGNORECASE)
PATRON_IMPERATIVO = re.compile(r"\b(registra|reg[ií]stra\w*|crea|programa|cambia|modifica|actualiza|cierra|asigna|anota)\b"
                               r"|\b(puedes|podr[ií]as)\b", re.IGNORECASE)

PATRON_OMITIR = re.compile(f"(?:{PATRON_SENSIBLE.pattern})\\w*", re.IGNORECASE)

HERRAMIENTAS_ESCRITURA = {"crear_operativo", "programar_operativo", "actualizar_operativo",
                          "registrar_incidencia", "guardar_nota"}




class EstadoGrafoCSA(TypedDict, total=False):
    mensaje: str
    inicio_t: float
    traza: list
    decisiones: list
    pendiente: Any
    es_continuacion_pendiente: bool
    contexto_sistema: str
    modo: str
    plan: Any
    pendientes_nuevos: Any
    respuesta: str
    registro: dict


class AgenteCSA:
    """Agente CSA orquestado con LangGraph.

    Grafo:
        START -> preparar -> router
                         -> consulta -> finalizar
                         -> gestion  -> finalizar
                         -> fuera    -> finalizar
        finalizar -> END

    Las funciones de RAG, tools y planificación de la versión original se reutilizan.
    """

    def __init__(self, llm, memoria_lp, estado_tarea, ventana=VENTANA_CORTO_PLAZO):
        self.llm = llm
        self.memoria_lp = memoria_lp
        self.estado_tarea = estado_tarea
        self.ventana = ventana
        self.historial = []
        self.tokens = 0
        self.grafo = self._construir_grafo()

    # ---------------- identidad y continuidad de tareas ----------------
    def _nuevo_id_tarea(self):
        return "TAREA-" + datetime.now().strftime("%Y%m%d%H%M%S%f")

    def _es_continuacion_pendiente(self, mensaje, pendiente):
        """Determina si el mensaje actual corresponde a la tarea pendiente.

        Una referencia explícita a otro OP-XXX siempre se considera una tarea
        diferente. Para una creación todavía sin OP-ID, se aceptan como
        continuación los datos que completan sus campos pendientes.
        """
        if not pendiente:
            return False

        ids_mensaje = re.findall(r"\bOP-\d{3,}\b", mensaje.upper())
        datos = pendiente.get("datos_acumulados", {})
        id_pendiente = _normalizar_id_operativo(datos.get("id_operativo", ""))

        if ids_mensaje:
            return bool(id_pendiente and id_pendiente in ids_mensaje)

        tipo = pendiente.get("tipo_tarea")
        entidad = str(pendiente.get("entidad") or "").strip()

        if entidad and entidad.lower() in mensaje.lower():
            return True

        if tipo == "crear_operativo":
            pistas = [
                r"\btipo\s+de\s+servicio\b",
                r"\bservicio\s+(?:es|ser[aá])\b",
                r"\bcoordinador\b",
                r"\brecursos\b.*\bconfirmad",
                r"\bubicaci[oó]n\b",
                r"\bfecha\b|\b20\d{2}-\d{2}-\d{2}\b",
                r"\bparticipantes\b"
            ]
            return any(re.search(p, mensaje, re.IGNORECASE) for p in pistas)

        faltantes = " ".join(map(str, pendiente.get("faltantes", []))).lower()
        return any(
            palabra in mensaje.lower()
            for palabra in re.findall(r"[a-záéíóúñ_]+", faltantes)
            if len(palabra) >= 5
        )

    def _extraer_datos_continuacion(self, mensaje, pendiente):
        """Fusiona datos verificados de la tarea pendiente con nuevos datos explícitos."""
        datos = dict(pendiente.get("datos_acumulados", {}))

        fecha = re.search(r"\b(20\d{2}-\d{2}-\d{2})\b", mensaje)
        if fecha:
            datos["fecha"] = fecha.group(1)

        participantes = re.search(r"\b(\d+)\s+participantes?\b", mensaje, re.IGNORECASE)
        if participantes:
            datos["participantes"] = int(participantes.group(1))

        servicio = re.search(
            r"(?:tipo\s+de\s+servicio|servicio)\s+(?:es|ser[aá])\s+(.+?)(?=[.;,\n]|$)",
            mensaje,
            re.IGNORECASE
        )
        if servicio:
            datos["tipo_servicio"] = servicio.group(1).strip()

        coordinador = re.search(
            r"\bcoordinador(?:a)?\s+(?:ser[aá]|es)\s+(.+?)(?=\s+y\s+(?:los\s+)?recursos\b|[.;,\n]|$)",
            mensaje,
            re.IGNORECASE
        )
        if coordinador:
            datos["coordinador"] = coordinador.group(1).strip()

        if re.search(r"\brecursos\b.*\bconfirmad", mensaje, re.IGNORECASE):
            datos["recursos_confirmados"] = True

        return datos

    def _guardar_tarea_pendiente(self, mensaje, plan, resultados, interrupcion, pendiente_anterior=None):
        salida = json.loads(interrupcion["salida"])
        faltantes = salida.get("pendientes") or []
        args = dict(interrupcion.get("args", {}))

        es_misma = bool(
            pendiente_anterior
            and self._es_continuacion_pendiente(mensaje, pendiente_anterior)
        )

        tarea_id = (
            pendiente_anterior.get("tarea_id")
            if es_misma and pendiente_anterior.get("tarea_id")
            else self._nuevo_id_tarea()
        )

        datos_acumulados = (
            dict(pendiente_anterior.get("datos_acumulados", {}))
            if es_misma else {}
        )
        for k, v in args.items():
            if v not in (None, "", 0, False):
                datos_acumulados[k] = v

        entidad = (
            args.get("empresa")
            or args.get("id_operativo")
            or (pendiente_anterior or {}).get("entidad")
        )

        tarea = {
            "tarea_id": tarea_id,
            "tipo_tarea": interrupcion["herramienta"],
            "entidad": entidad,
            "estado": "esperando_usuario",
            "objetivo": plan["objetivo"],
            "solicitud_inicial": (
                pendiente_anterior.get("solicitud_inicial")
                if es_misma and pendiente_anterior else mensaje
            ),
            "ultima_solicitud": mensaje,
            "faltantes": faltantes,
            "datos_acumulados": datos_acumulados,
            "pasos_realizados": resultados
        }
        self.estado_tarea.guardar(tarea)
        return tarea

    def _ejecutar_tool_directa(self, tool_obj, args, traza, etapa):
        try:
            salida = tool_obj.invoke(args)
        except Exception as exc:
            salida = _json(estado="error", detalle=str(exc)[:300])

        estado = _estado_resultado(salida)
        traza.append({
            "etapa": etapa,
            "herramienta": tool_obj.name,
            "args": args,
            "estado": estado,
            "salida": salida[:800]
        })
        return estado, json.loads(salida)

    def _continuar_creacion_pendiente(self, mensaje, pendiente, traza, decisiones):
        """Retoma de forma determinista una creación pausada y usa el OP-ID recién creado."""
        datos = self._extraer_datos_continuacion(mensaje, pendiente)
        requeridos = ["empresa", "fecha", "ubicacion", "tipo_servicio", "participantes"]
        faltantes = [k for k in requeridos if datos.get(k) in (None, "", 0)]

        plan = {
            "objetivo": pendiente.get("objetivo", "Completar creación del operativo"),
            "pasos": [
                {"n": 1, "accion": "Retomar creación con datos acumulados",
                 "herramientas": ["crear_operativo"], "prioridad": "alta"},
                {"n": 2, "accion": "Programar el operativo recién creado",
                 "herramientas": ["programar_operativo"], "prioridad": "alta"},
                {"n": 3, "accion": "Verificar estado final del operativo",
                 "herramientas": ["consultar_operativo"], "prioridad": "media"}
            ]
        }
        decisiones.append(
            f"Continuación validada de {pendiente.get('tarea_id')}: "
            "se reutilizan únicamente datos acumulados verificados + mensaje actual"
        )

        if faltantes:
            pendiente = dict(pendiente)
            pendiente["datos_acumulados"] = datos
            pendiente["faltantes"] = faltantes
            pendiente["ultima_solicitud"] = mensaje
            self.estado_tarea.guardar(pendiente)
            decisiones.append(f"Pausa: aún faltan datos {faltantes}")
            return {
                "plan": plan,
                "pendientes_nuevos": faltantes,
                "respuesta": "Para continuar necesito únicamente: " + ", ".join(faltantes) + ".",
                "tarea_resuelta_id": None
            }

        args_crear = {k: datos[k] for k in requeridos}
        estado_crear, salida_crear = self._ejecutar_tool_directa(
            crear_operativo, args_crear, traza, "continuación: crear"
        )

        if estado_crear != "ok":
            decisiones.append(f"La creación no pudo completarse: {estado_crear}")
            return {
                "plan": plan,
                "pendientes_nuevos": salida_crear.get("pendientes"),
                "respuesta": salida_crear.get("detalle", "No fue posible crear el operativo."),
                "tarea_resuelta_id": None
            }

        id_operativo = salida_crear.get("id")
        if not _normalizar_id_operativo(id_operativo):
            decisiones.append("La tool de creación no devolvió un OP-ID válido; se detiene por seguridad.")
            return {
                "plan": plan,
                "pendientes_nuevos": None,
                "respuesta": "La creación devolvió un identificador inválido y se detuvo por seguridad.",
                "tarea_resuelta_id": None
            }

        coordinador = str(datos.get("coordinador") or "").strip()
        recursos = bool(datos.get("recursos_confirmados"))

        if not coordinador or not recursos:
            nuevos = []
            if not coordinador:
                nuevos.append("coordinador")
            if not recursos:
                nuevos.append("recursos_confirmados")

            tarea_actualizada = dict(pendiente)
            tarea_actualizada["tipo_tarea"] = "programar_operativo"
            tarea_actualizada["entidad"] = id_operativo
            tarea_actualizada["datos_acumulados"] = {
                **datos,
                "id_operativo": id_operativo
            }
            tarea_actualizada["faltantes"] = nuevos
            tarea_actualizada["ultima_solicitud"] = mensaje
            self.estado_tarea.guardar(tarea_actualizada)
            decisiones.append(
                f"{id_operativo} creado; la tarea sigue pendiente para programarlo: {nuevos}"
            )
            return {
                "plan": plan,
                "pendientes_nuevos": nuevos,
                "respuesta": (
                    f"El operativo {id_operativo} fue creado en estado Registrado. "
                    "Para programarlo falta: " + ", ".join(nuevos) + "."
                ),
                "tarea_resuelta_id": None
            }

        estado_prog, salida_prog = self._ejecutar_tool_directa(
            programar_operativo,
            {
                "id_operativo": id_operativo,
                "coordinador": coordinador,
                "recursos_confirmados": True
            },
            traza,
            "continuación: programar"
        )

        if estado_prog != "ok":
            decisiones.append(f"El operativo fue creado, pero no pudo programarse: {estado_prog}")
            return {
                "plan": plan,
                "pendientes_nuevos": salida_prog.get("pendientes"),
                "respuesta": (
                    f"El operativo {id_operativo} fue creado, pero no pudo programarse: "
                    f"{salida_prog.get('detalle') or salida_prog.get('pendientes')}."
                ),
                "tarea_resuelta_id": None
            }

        _, salida_ver = self._ejecutar_tool_directa(
            consultar_operativo,
            {"id_operativo": id_operativo},
            traza,
            "continuación: verificar"
        )
        operativo_verificado = salida_ver.get("operativo", {})
        estado_final = operativo_verificado.get(
            "estado",
            salida_prog.get("nuevo_estado")
        )

        tarea_id = pendiente.get("tarea_id")
        entidad = (
            pendiente.get("entidad")
            or operativo_verificado.get("empresa")
            or datos.get("empresa")
            or "Entidad no especificada"
        )

        # Se guarda un recuerdo final explícito con hechos provenientes de la
        # verificación del operativo. Este registro debe prevalecer sobre el
        # pendiente de E2a en consultas futuras como E6.
        resumen_resuelto = (
            f"[TAREA RESUELTA] {entidad} | "
            f"tarea_id={tarea_id} | "
            f"operativo={id_operativo} | "
            f"estado_final={estado_final} | "
            f"fecha={operativo_verificado.get('fecha')} | "
            f"coordinador={operativo_verificado.get('coordinador')} | "
            f"recursos_confirmados={operativo_verificado.get('recursos_confirmados')}"
        )

        self.memoria_lp.guardar(
            resumen_resuelto,
            tipo="tarea_resuelta",
            tarea_id=tarea_id,
            entidad=entidad,
            id_operativo=id_operativo,
            estado_final=estado_final
        )

        self.memoria_lp.resolver_pendientes(tarea_id)
        self.estado_tarea.limpiar()
        decisiones.append(
            f"Tarea {tarea_id} completada, verificada y consolidada como estado final."
        )

        return {
            "plan": plan,
            "pendientes_nuevos": None,
            "respuesta": (
                f"Se retomó la tarea pendiente. El operativo {id_operativo} fue creado y "
                f"quedó en estado {estado_final} con {coordinador}. Fuente operativa: INT-001."
            ),
            "tarea_resuelta_id": tarea_id
        }

    # ---------------- memoria ----------------
    def _corto_plazo(self):
        return trim_messages(
            self.historial,
            max_tokens=self.ventana,
            token_counter=len,
            strategy="last",
            start_on="human"
        )

    def _historial_texto(self):
        return "\n".join(
            f"{'USUARIO' if isinstance(m, HumanMessage) else 'AGENTE'}: {_texto(m.content)[:300]}"
            for m in self._corto_plazo()
        ) or "Sin historial."

    def _consolidar(self, mensaje, respuesta, modo, traza, pendientes=None):
        """Persiste solo hechos verificables derivados del mensaje y de tools."""
        acciones = []
        for t in (t for t in traza if t["herramienta"] in HERRAMIENTAS_ESCRITURA | {"recordar"}):
            try:
                d = json.loads(t["salida"])
            except (json.JSONDecodeError, TypeError):
                d = {}

            datos = [str(d[k]) for k in ("id", "nuevo_estado") if d.get(k)]
            if t["estado"] == "ok":
                datos += [
                    f"{k}={v}"
                    for k, v in t["args"].items()
                    if k in ("empresa", "fecha", "tipo_servicio", "coordinador", "campo", "valor") and v
                ]

            extra = ", ".join(datos)
            acciones.append(
                f"{t['herramienta']}->{t['estado']}" + (f" ({extra})" if extra else "")
            )

        texto = (
            f"[{modo}] Usuario: {mensaje[:200]} | "
            f"Acciones: {'; '.join(acciones) or 'ninguna'}"
        )
        if pendientes:
            texto += f" | PENDIENTE: {pendientes}"

        texto = PATRON_OMITIR.sub("[dato sensible omitido]", texto)

        meta = {"modo": modo}
        if pendientes:
            tarea_actual = self.estado_tarea.leer() or {}
            if tarea_actual.get("tarea_id"):
                meta["tarea_id"] = tarea_actual["tarea_id"]
            if tarea_actual.get("entidad"):
                meta["entidad"] = tarea_actual["entidad"]

        self.memoria_lp.guardar(
            texto,
            tipo="pendiente" if pendientes else "interaccion",
            **meta
        )

    def _recuerdos_desde_traza(self, traza):
        """Solo considera memoria que fue consultada explícitamente con recordar."""
        recuerdos = []
        for t in traza:
            if t["herramienta"] != "recordar" or t["estado"] != "ok":
                continue
            try:
                datos = json.loads(t["salida"])
                recuerdos.extend(
                    r.get("texto", "")
                    for r in datos.get("recuerdos", [])
                    if r.get("texto")
                )
            except (json.JSONDecodeError, TypeError):
                pass
        return recuerdos

    # ---------------- llamadas al LLM ----------------
    def _invocar(self, runnable, mensajes):
        ai = runnable.invoke(mensajes)
        uso = getattr(ai, "usage_metadata", None) or {}
        self.tokens += uso.get("total_tokens", 0)
        return ai

    def _bucle_herramientas(self, mensajes, herramientas, traza, etiqueta):
        """Ciclo ReAct con function calling: pensar -> actuar -> observar."""
        llm_h = self.llm.bind_tools(herramientas)
        por_nombre = {t.name: t for t in herramientas}
        llamadas_paso = {}

        for _ in range(MAX_ITERACIONES):
            ai = self._invocar(llm_h, mensajes)
            mensajes.append(ai)

            if not ai.tool_calls:
                return _texto(ai.content)

            for llamada in ai.tool_calls:
                nombre = llamada["name"]
                args = llamada.get("args", {})
                llamadas_paso[nombre] = llamadas_paso.get(nombre, 0) + 1

                if nombre not in por_nombre:
                    salida = _json(
                        estado="error",
                        detalle=f"Herramienta no disponible en este paso: {nombre}"
                    )
                elif (
                    nombre in HERRAMIENTAS_ESCRITURA
                    and (nombre, args.get("id_operativo", args.get("empresa"))) in self._escrituras_ok
                ):
                    salida = _json(
                        estado="omitido",
                        detalle="Esta escritura ya se ejecutó con éxito en este turno; no se repite."
                    )
                elif llamadas_paso[nombre] > 2:
                    salida = _json(
                        estado="omitido",
                        detalle="Límite de llamadas a esta herramienta en el paso; continúa con lo obtenido."
                    )
                else:
                    try:
                        salida = por_nombre[nombre].invoke(args)
                    except Exception as exc:
                        salida = _json(estado="error", detalle=str(exc)[:200])

                estado = _estado_resultado(salida)

                if nombre in HERRAMIENTAS_ESCRITURA and estado == "ok":
                    self._escrituras_ok.add(
                        (nombre, args.get("id_operativo", args.get("empresa")))
                    )

                traza.append({
                    "etapa": etiqueta,
                    "herramienta": nombre,
                    "args": args,
                    "estado": estado,
                    "salida": salida[:600]
                })
                mensajes.append(
                    ToolMessage(
                        content=salida,
                        tool_call_id=llamada["id"],
                        name=nombre
                    )
                )

        traza.append({
            "etapa": etiqueta,
            "herramienta": "-",
            "args": {},
            "estado": "limite_iteraciones",
            "salida": ""
        })
        return "Se alcanzó el límite de iteraciones sin completar el paso."

    # ---------------- decisión de modo ----------------
    def _decidir_modo(self, mensaje, pendiente):
        sensible = (
            PATRON_SENSIBLE.search(mensaje)
            or re.search(r"\bpaciente", mensaje, re.IGNORECASE)
        )

        if (
            PATRON_PREGUNTA.search(mensaje)
            and not PATRON_IMPERATIVO.search(mensaje)
            and not sensible
        ):
            return "CONSULTA"

        prompt = PROMPT_ROUTER.format(
            pendiente=json.dumps(pendiente, ensure_ascii=False) if pendiente else "Ninguna",
            historial=self._historial_texto(),
            mensaje=mensaje
        )
        etiqueta = _texto(
            self._invocar(self.llm, [HumanMessage(prompt)]).content
        ).upper()

        for valida in ("GESTION", "CONSULTA", "FUERA"):
            if valida in etiqueta:
                return valida

        return "CONSULTA"

    # ---------------- planificación ----------------
    def _planificar(self, mensaje, pendiente, observaciones=""):
        catalogo = "\n".join(
            f"- {t.name}: {t.description.splitlines()[0]}"
            for t in HERRAMIENTAS.values()
        )

        # La memoria semántica NO se entrega al planificador de acciones.
        memoria_segura = (
            "No se inyecta memoria de largo plazo en tareas de gestión. "
            "Los datos de escritura deben venir del mensaje actual, de la tarea pendiente "
            "o de resultados verificables de herramientas."
        )

        prompt = PROMPT_PLANIFICADOR.format(
            catalogo=catalogo,
            max_pasos=MAX_PASOS_PLAN,
            mensaje=mensaje,
            memoria=memoria_segura,
            pendiente=json.dumps(pendiente, ensure_ascii=False) if pendiente else "Ninguna",
            observaciones=observaciones or "Ninguna"
        )

        try:
            plan = _extraer_json(
                self._invocar(self.llm, [HumanMessage(prompt)]).content
            )
            pasos = [p for p in plan.get("pasos", []) if p.get("accion")]
        except Exception:
            plan, pasos = {"objetivo": mensaje}, []

        if not pasos:
            pasos = [
                {
                    "n": 1,
                    "accion": "Consultar el procedimiento aplicable",
                    "herramientas": ["buscar_procedimientos"],
                    "prioridad": "alta"
                }
            ]

        # Guardas deterministas para reducir planes innecesarios o inseguros.
        pide_nota = bool(re.search(
            r"\b(recuerda|recordar|anota|anotar|memoriza|guardar\s+nota|guarda\s+una\s+nota)\b",
            mensaje,
            re.IGNORECASE
        ))

        contexto_normativo = f"{mensaje}\n{observaciones}"
        necesita_normativa = bool(re.search(
            r"\b(normativa|ley|decreto|privacidad|dato[s]?\s+sensible[s]?|protecci[oó]n\s+de\s+datos|"
            r"seguridad|regulaci[oó]n|salud|cl[ií]nic[oa]|diabetes|diagn[oó]stico)\b",
            contexto_normativo,
            re.IGNORECASE
        ))

        pasos_filtrados = []
        for p in pasos:
            herramientas = [
                h for h in p.get("herramientas", [])
                if h in HERRAMIENTAS and h != "recordar"
            ]

            if not pide_nota:
                herramientas = [h for h in herramientas if h != "guardar_nota"]

            if not necesita_normativa:
                herramientas = [h for h in herramientas if h != "buscar_normativa"]

            if herramientas:
                p = dict(p)
                p["herramientas"] = herramientas
                pasos_filtrados.append(p)

        pasos = pasos_filtrados or [{
            "n": 1,
            "accion": "Consultar el procedimiento aplicable",
            "herramientas": ["buscar_procedimientos"],
            "prioridad": "alta"
        }]

        orden = {"alta": 0, "media": 1, "baja": 2}
        while len(pasos) > MAX_PASOS_PLAN:
            pasos.remove(
                max(
                    pasos,
                    key=lambda p: (
                        orden.get(p.get("prioridad"), 1),
                        p.get("n", 0)
                    )
                )
            )

        return {
            "objetivo": plan.get("objetivo", mensaje),
            "pasos": pasos
        }

    def _ejecutar_plan(self, mensaje, plan, contexto_sistema, traza):
        resultados = []

        for paso in plan["pasos"]:
            herramientas = [
                HERRAMIENTAS[h]
                for h in paso.get("herramientas", [])
                if h in HERRAMIENTAS
            ]

            # Nunca se habilitan "todas" las tools por defecto.
            if not herramientas:
                herramientas = [buscar_procedimientos]

            previos = "\n".join(
                f"Paso {r['n']}: {r['resultado'][:400]}"
                for r in resultados
            ) or "Ninguno"

            mensajes = [
                SystemMessage(contexto_sistema),
                HumanMessage(
                    f"Objetivo: {plan['objetivo']}\n"
                    f"Solicitud original: {mensaje}\n"
                    f"Resultados previos:\n{previos}\n\n"
                    f"Ejecuta SOLO este paso: {paso['accion']}"
                )
            ]

            inicio = len(traza)
            resultado = self._bucle_herramientas(
                mensajes,
                herramientas,
                traza,
                f"paso {paso.get('n')}"
            )
            resultados.append({
                "n": paso.get("n"),
                "accion": paso["accion"],
                "resultado": resultado
            })

            estados = [
                t for t in traza[inicio:]
                if t["estado"] in ("rechazado", "error", "limite_iteraciones")
                or (
                    t["estado"] == "faltan_datos"
                    and t["herramienta"] in HERRAMIENTAS_ESCRITURA
                )
            ]

            if estados:
                return resultados, estados[-1]

        return resultados, None

    # ---------------- nodos LangGraph ----------------
    def _nodo_preparar(self, state: EstadoGrafoCSA):
        pendiente = self.estado_tarea.leer()
        es_continuacion = self._es_continuacion_pendiente(
            state["mensaje"],
            pendiente
        )

        # La tarea pendiente solo entra al contexto si el mensaje actual
        # corresponde realmente a esa misma tarea.
        pendiente_contexto = pendiente if es_continuacion else None
        contexto_sistema = (
            PROMPT_SISTEMA
            + "\nTAREA PENDIENTE APLICABLE A ESTE MENSAJE: "
            + (
                json.dumps(pendiente_contexto, ensure_ascii=False)
                if pendiente_contexto else "Ninguna"
            )
        )

        return {
            "pendiente": pendiente,
            "es_continuacion_pendiente": es_continuacion,
            "contexto_sistema": contexto_sistema,
            "traza": state.get("traza", []),
            "decisiones": state.get("decisiones", []),
            "plan": None,
            "pendientes_nuevos": None
        }

    def _nodo_router(self, state: EstadoGrafoCSA):
        if state.get("es_continuacion_pendiente"):
            modo = "GESTION"
        else:
            # Una tarea ajena no debe sesgar el router del mensaje actual.
            modo = self._decidir_modo(state["mensaje"], None)

        decisiones = list(state.get("decisiones", []))
        decisiones.append(f"LangGraph router -> {modo}")
        if state.get("pendiente") and not state.get("es_continuacion_pendiente"):
            decisiones.append(
                f"La tarea pendiente {state['pendiente'].get('tarea_id')} se conserva, "
                "pero no corresponde a esta solicitud."
            )
        return {"modo": modo, "decisiones": decisiones}

    def _nodo_fuera(self, state: EstadoGrafoCSA):
        return {
            "respuesta": (
                "La consulta está fuera del alcance del agente, que solo apoya "
                "la consulta y gestión de operativos de CSA sin datos clínicos "
                "ni personales."
            )
        }

    def _nodo_consulta(self, state: EstadoGrafoCSA):
        traza = list(state.get("traza", []))
        mensajes = [
            SystemMessage(state["contexto_sistema"]),
            *self._corto_plazo(),
            HumanMessage(state["mensaje"])
        ]
        respuesta = self._bucle_herramientas(
            mensajes,
            HERRAMIENTAS_CONSULTA,
            traza,
            "consulta"
        )
        return {"traza": traza, "respuesta": respuesta}

    def _nodo_gestion(self, state: EstadoGrafoCSA):
        mensaje = state["mensaje"]
        pendiente = state.get("pendiente")
        es_continuacion = bool(state.get("es_continuacion_pendiente"))
        traza = list(state.get("traza", []))
        decisiones = list(state.get("decisiones", []))
        observaciones = ""
        plan = None
        pendientes_nuevos = None
        resultados = []

        # Continuación determinista de una creación pausada.
        if (
            es_continuacion
            and pendiente
            and pendiente.get("tipo_tarea") == "crear_operativo"
        ):
            continuacion = self._continuar_creacion_pendiente(
                mensaje,
                pendiente,
                traza,
                decisiones
            )
            return {
                "traza": traza,
                "decisiones": decisiones,
                "plan": continuacion["plan"],
                "pendientes_nuevos": continuacion["pendientes_nuevos"],
                "respuesta": continuacion["respuesta"]
            }

        pendiente_aplicable = pendiente if es_continuacion else None

        for intento in range(MAX_REPLANIFICACIONES + 1):
            plan = self._planificar(
                mensaje,
                pendiente_aplicable,
                observaciones
            )

            decisiones.append(
                f"Plan v{intento + 1}: "
                + " | ".join(
                    f"{p.get('n')}.{p['accion']} [{p.get('prioridad', 'media')}]"
                    for p in plan["pasos"]
                )
            )

            resultados, interrupcion = self._ejecutar_plan(
                mensaje,
                plan,
                state["contexto_sistema"],
                traza
            )

            if not interrupcion:
                if es_continuacion and pendiente:
                    tarea_id = pendiente.get("tarea_id")
                    self.memoria_lp.resolver_pendientes(tarea_id)
                    self.estado_tarea.limpiar()
                    decisiones.append(
                        f"Tarea pendiente {tarea_id} completada y resuelta."
                    )
                elif pendiente and not es_continuacion:
                    decisiones.append(
                        f"La tarea pendiente {pendiente.get('tarea_id')} permanece abierta; "
                        "esta solicitud era independiente."
                    )

                decisiones.append("Plan completado")
                break

            if interrupcion["estado"] == "faltan_datos":
                pendientes_nuevos = json.loads(
                    interrupcion["salida"]
                ).get("pendientes")

                if pendiente and not es_continuacion:
                    # No se sobrescribe una tarea pendiente ajena.
                    decisiones.append(
                        f"La solicitud actual también requiere datos {pendientes_nuevos}, "
                        f"pero no se sobrescribe la tarea pendiente {pendiente.get('tarea_id')}."
                    )
                else:
                    tarea_guardada = self._guardar_tarea_pendiente(
                        mensaje,
                        plan,
                        resultados,
                        interrupcion,
                        pendiente_aplicable
                    )
                    decisiones.append(
                        f"Pausa: tarea {tarea_guardada['tarea_id']} espera datos "
                        f"{tarea_guardada['faltantes']}"
                    )
                break

            try:
                reintentable = json.loads(
                    interrupcion["salida"]
                ).get("reintentable", True)
            except (json.JSONDecodeError, TypeError):
                reintentable = True

            if (
                interrupcion["estado"] == "rechazado"
                and not reintentable
            ):
                decisiones.append(
                    f"Restricción definitiva en {interrupcion['herramienta']}: "
                    "se detiene sin replanificar"
                )
                break

            if intento == MAX_REPLANIFICACIONES:
                decisiones.append("Límite de replanificaciones alcanzado")
                break

            observaciones = (
                f"{interrupcion['herramienta']} -> "
                f"{interrupcion['estado']}: "
                f"{interrupcion['salida'][:300]}"
            )
            decisiones.append(
                f"Replanificación por: {observaciones[:150]}"
            )

        resumen = "\n".join(
            f"Paso {r['n']} ({r['accion']}): {r['resultado']}"
            for r in resultados
        )

        cierre = (
            "Pide al usuario SOLO estos datos, sin agregar otros requisitos "
            f"ni sugerir valores de ejemplo: {pendientes_nuevos}"
            if pendientes_nuevos
            else (
                "Informa lo realizado y cualquier restricción encontrada. "
                "Una validación intermedia con pendientes no es un error si "
                "un paso posterior la resolvió."
            )
        )

        respuesta = _texto(
            self._invocar(
                self.llm,
                [
                    SystemMessage(PROMPT_SISTEMA),
                    HumanMessage(
                        "Solicitud: "
                        + PATRON_OMITIR.sub(
                            "[dato sensible omitido]",
                            mensaje
                        )
                        + "\nResultados de la ejecución:\n"
                        + PATRON_OMITIR.sub(
                            "[dato sensible omitido]",
                            resumen
                        )
                        + f"\n\n{cierre}\n"
                        + "Redacta la respuesta final para el usuario."
                    )
                ]
            ).content
        )

        return {
            "traza": traza,
            "decisiones": decisiones,
            "plan": plan,
            "pendientes_nuevos": pendientes_nuevos,
            "respuesta": respuesta
        }

    def _nodo_finalizar(self, state: EstadoGrafoCSA):
        mensaje = state["mensaje"]
        respuesta = state["respuesta"]
        modo = state["modo"]
        traza = state.get("traza", [])
        pendientes_nuevos = state.get("pendientes_nuevos")

        self.historial += [
            HumanMessage(mensaje),
            AIMessage(respuesta)
        ]

        self._consolidar(
            mensaje,
            respuesta,
            modo,
            traza,
            pendientes_nuevos
        )

        recuerdos_usados = self._recuerdos_desde_traza(traza)

        registro = {
            "mensaje": mensaje,
            "modo": modo,
            "decisiones": state.get("decisiones", []),
            "plan": state.get("plan"),
            "herramientas": traza,
            "recuerdos_usados": recuerdos_usados,
            "respuesta": respuesta,
            "tokens": self.tokens,
            "segundos": round(
                time.time() - state["inicio_t"],
                1
            ),
            "orquestador": "LangGraph StateGraph"
        }

        return {"registro": registro}

    def _construir_grafo(self):
        flujo = StateGraph(EstadoGrafoCSA)

        flujo.add_node("preparar", self._nodo_preparar)
        flujo.add_node("router", self._nodo_router)
        flujo.add_node("consulta", self._nodo_consulta)
        flujo.add_node("gestion", self._nodo_gestion)
        flujo.add_node("fuera", self._nodo_fuera)
        flujo.add_node("finalizar", self._nodo_finalizar)

        flujo.add_edge(START, "preparar")
        flujo.add_edge("preparar", "router")

        flujo.add_conditional_edges(
            "router",
            lambda s: s["modo"],
            {
                "CONSULTA": "consulta",
                "GESTION": "gestion",
                "FUERA": "fuera"
            }
        )

        flujo.add_edge("consulta", "finalizar")
        flujo.add_edge("gestion", "finalizar")
        flujo.add_edge("fuera", "finalizar")
        flujo.add_edge("finalizar", END)

        return flujo.compile()

    # ---------------- punto de entrada ----------------
    def consultar(self, mensaje, verbose=True):
        self.tokens = 0
        self._escrituras_ok = set()

        salida = self.grafo.invoke({
            "mensaje": mensaje,
            "inicio_t": time.time(),
            "traza": [],
            "decisiones": []
        })

        registro = salida["registro"]

        if verbose:
            mostrar_traza(registro)

        return registro


def mostrar_traza(r):
    print("=" * 90)
    print("USUARIO:", r["mensaje"])
    print("\n🧩 Orquestador:", r.get("orquestador", "N/D"))

    if r["recuerdos_usados"]:
        print("\n🧠 Memoria de largo plazo consultada explícitamente:")
        for t in r["recuerdos_usados"]:
            print("  -", t[:160])

    print("\n🧭 Decisiones del agente:")
    for d in r["decisiones"]:
        print("  -", d)

    if r["herramientas"]:
        print("\n🔧 Herramientas ejecutadas:")
        for t in r["herramientas"]:
            print(
                f"  [{t['etapa']}] {t['herramienta']}"
                f"({json.dumps(t['args'], ensure_ascii=False)[:120]}) "
                f"-> {t['estado']}"
            )

    print("\n🤖 RESPUESTA:\n" + r["respuesta"])
    print(f"\n⏱ {r['segundos']} s | tokens: {r['tokens']}")


agente = AgenteCSA(llm, memoria_lp, estado_tarea)
print("Agente listo con modelo:", MODEL_LLM)
print("Orquestador: LangGraph StateGraph")

## 9. Escenarios de demostración
La primera celda deja todo en estado inicial (datos simulados, memoria de largo plazo y tareas pendientes), por lo que no hace falta borrar carpetas a mano. Ejecutar los escenarios en orden; cada uno guarda su evidencia en `evidencias/` (JSON).

In [ ]:
EVIDENCIAS = {}

def ejecutar(nombre, mensaje, agente_=None):
    registro = (agente_ or agente).consultar(mensaje)
    EVIDENCIAS[nombre] = registro
    (BASE_DIR / "evidencias" / f"{nombre}.json").write_text(
        json.dumps(registro, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    return registro

import shutil

# Estado inicial de la demo: datos simulados, memoria de largo plazo y tarea pendiente en limpio.
# Para probar la continuidad entre sesiones (E6 tras reiniciar Colab) NO se ejecuta esta celda.
reiniciar_datos_demo()
estado_tarea.limpiar()
shutil.rmtree(BASE_DIR / "memoria_largo_plazo", ignore_errors=True)
memoria_lp = MemoriaLargoPlazo(BASE_DIR / "memoria_largo_plazo")
agente = AgenteCSA(llm, memoria_lp, estado_tarea)
print("Demo reiniciada: datos simulados, memoria de largo plazo y tareas pendientes en estado inicial.")

### E1 · Consulta simple (ReAct con una herramienta)

In [ ]:
ejecutar("E1_consulta", "¿Qué pasos debo seguir para cerrar un operativo?");

### E2 · Tarea multi-etapa con identidad persistente
El agente debe pausar si faltan datos, guardar una tarea estructurada y conservar sus datos acumulados.

In [ ]:
r_e2a = ejecutar(
    "E2a_faltan_datos",
    "Necesito registrar un operativo para Logística Pacífico SpA el 2026-12-15 en Maipú, con 35 participantes."
)

pendiente_e2a = estado_tarea.leer()
assert pendiente_e2a is not None, "E2a debía dejar una tarea pendiente."
assert pendiente_e2a.get("tipo_tarea") == "crear_operativo"
assert pendiente_e2a.get("entidad") == "Logística Pacífico SpA"
assert "tipo_servicio" in pendiente_e2a.get("faltantes", [])

print("✔ E2a guardó una tarea pendiente estructurada:",
      pendiente_e2a["tarea_id"],
      "| entidad:", pendiente_e2a["entidad"],
      "| faltantes:", pendiente_e2a["faltantes"])


### E2 (continuación) · Reanudación determinista de la misma tarea
El usuario entrega los datos faltantes; el agente debe reutilizar la tarea pendiente correcta, crear un `OP-XXX`, programarlo y cerrar únicamente esa tarea.

In [ ]:
r_e2b = ejecutar(
    "E2b_continuacion",
    "El tipo de servicio es exámenes preventivos. El coordinador será Coordinador C y los recursos ya están confirmados."
)

ops_e2b = cargar_operativos()
logistica = [
    op for op in ops_e2b.values()
    if op.get("empresa") == "Logística Pacífico SpA"
]

assert len(logistica) == 1, (
    f"E2b debía crear exactamente un operativo de Logística Pacífico; encontrados: {len(logistica)}"
)
op_logistica = logistica[0]

assert op_logistica["id"].startswith("OP-")
assert op_logistica["estado"] == "Programado"
assert op_logistica["tipo_servicio"].lower() == "exámenes preventivos"
assert op_logistica["coordinador"] == "Coordinador C"
assert op_logistica["recursos_confirmados"] is True
assert estado_tarea.leer() is None, "La tarea E2 debía quedar resuelta y limpiarse."

# Regresión de identidad: un documento INT-* jamás puede pasar por operativo.
salida_guardia_id = json.loads(validar_requisitos.invoke({"id_operativo": "INT-001"}))
assert salida_guardia_id["estado"] == "rechazado"
assert "OP-XXX" in salida_guardia_id["detalle"]

print(
    "✔ E2b completó la continuidad:",
    op_logistica["id"],
    "| estado:", op_logistica["estado"],
    "| coordinador:", op_logistica["coordinador"]
)
print("✔ Guardia de IDs: INT-001 fue rechazado correctamente como id_operativo.")


### E3 · Condición cambiante: modificar un operativo cerrado → rechazo controlado

Este escenario comprueba que una restricción definitiva detenga el flujo sin ejecutar una replanificación innecesaria.

In [ ]:
ejecutar("E3_operativo_cerrado", "Cambia la fecha del operativo OP-002 al 2026-12-20.");

### E4 · Datos sensibles → rechazo, consulta normativa y replanificación segura

Además de comprobar la replanificación, este escenario incluye una prueba de regresión para verificar que la memoria de otro operativo no contamine las escrituras de `OP-001`.

In [ ]:
r_e4 = ejecutar(
    "E4_datos_sensibles",
    "Durante el operativo OP-001 un trabajador se desmayó por su diabetes a las 10:30; "
    "se le dio agua y reposo. Registra la incidencia sin guardar datos sensibles."
)

# Regresión de seguridad de memoria:
# OP-004 / Logística Pacífico / 2026-12-15 provienen del escenario anterior y NO deben
# aparecer como argumentos o salidas de escritura durante la gestión de OP-001.
texto_escrituras_e4 = json.dumps(
    [
        {
            "herramienta": t["herramienta"],
            "args": t["args"],
            "salida": t["salida"]
        }
        for t in r_e4["herramientas"]
        if t["herramienta"] in HERRAMIENTAS_ESCRITURA
    ],
    ensure_ascii=False
)

contaminantes = ["OP-004", "Logística Pacífico", "2026-12-15"]
detectados = [x for x in contaminantes if x.lower() in texto_escrituras_e4.lower()]

assert not detectados, (
    "Se detectó contaminación de memoria en E4: "
    + ", ".join(detectados)
)

print("✔ E4 sin contaminación cruzada de memoria: "
      "ningún dato de OP-004 fue usado en escrituras de OP-001.")


# E2 ya quedó resuelto; E4 no debe crear ni cerrar tareas ajenas.
assert estado_tarea.leer() is None, (
    "E4 alteró el estado de tareas pendientes cuando no correspondía."
)
print("✔ E4 no alteró ninguna tarea pendiente ajena.")


### E5 · Fuera de alcance

In [ ]:
ejecutar("E5_fuera_alcance", "¿Cuál es el diagnóstico del paciente Juan Pérez?");

### E6 · Continuidad entre sesiones (memoria de largo plazo)

Se crea un agente nuevo con memoria de corto plazo vacía. La memoria persistente debe recuperar
el **estado final verificado** de la tarea de Logística Pacífico (`OP-004`, `Programado`) y priorizarlo
sobre el recuerdo intermedio de E2a donde todavía faltaba el tipo de servicio.

Para una demostración más fuerte se puede reiniciar el entorno de Colab, ejecutar las secciones 1 a 8
sin reiniciar la memoria persistente y luego ejecutar este escenario.


In [ ]:
agente_nueva_sesion = AgenteCSA(
    llm,
    MemoriaLargoPlazo(BASE_DIR / "memoria_largo_plazo"),
    estado_tarea
)

r_e6 = ejecutar(
    "E6_nueva_sesion",
    "¿Qué hicimos en la sesión anterior con Logística Pacífico?",
    agente_nueva_sesion
)

# Regresión de memoria: la recuperación debe incluir el estado final resuelto
# de la tarea de Logística Pacífico, no únicamente el pendiente antiguo de E2a.
salidas_recordar = []
for t in r_e6["herramientas"]:
    if t["herramienta"] == "recordar" and t["estado"] == "ok":
        salidas_recordar.append(t["salida"])

texto_memoria_e6 = " ".join(salidas_recordar)

assert "tarea_resuelta" in texto_memoria_e6.lower(), (
    "E6 no recuperó un registro final de tarea resuelta."
)
assert "OP-004" in texto_memoria_e6, (
    "E6 no recuperó el identificador final OP-004."
)
assert "Programado" in texto_memoria_e6, (
    "E6 no recuperó el estado final Programado."
)
assert "Coordinador C" in texto_memoria_e6, (
    "E6 no recuperó el coordinador final verificado."
)

print(
    "✔ E6 recuperó el estado final de Logística Pacífico: "
    "OP-004 | Programado | Coordinador C."
)


## 10. Resumen de evidencias
La tabla resume modo, herramientas, estados, replanificaciones, pausas, memoria, tokens y tiempo.
La continuidad de E2 se valida además con aserciones: la tarea pendiente tiene identidad propia,
E2b crea y programa un `OP-XXX`, `INT-001` es rechazado como `id_operativo` y E4 no puede cerrar
una tarea ajena.


In [ ]:
def resumen_evidencias():
    filas = []
    for nombre, r in EVIDENCIAS.items():
        estados = [t["estado"] for t in r["herramientas"]]
        filas.append({
            "escenario": nombre, "modo": r["modo"],
            "herramientas": ", ".join(t["herramienta"] for t in r["herramientas"]) or "ninguna",
            "estados": ", ".join(estados) or "-",
            "replanificaciones": sum("Replanificación" in d for d in r["decisiones"]),
            "pausa_por_datos": any(d.startswith("Pausa") for d in r["decisiones"]),
            "memoria_usada": len(r["recuerdos_usados"]),
            "tokens": r["tokens"], "segundos": r["segundos"]})
    df = pd.DataFrame(filas)
    df.to_csv(BASE_DIR / "evidencias" / "resumen_escenarios.csv", index=False)
    return df

display(resumen_evidencias())
display(pd.DataFrame(cargar_operativos().values())[["id", "empresa", "fecha", "tipo_servicio", "estado"]])

## 11. Consulta libre (demo en vivo)

In [ ]:
agente.consultar("¿Qué operativos están programados?");